# pop pretrain -- Phase 2 span-corruption pretraining

GPU stage: trains the real (vocab 16384) SentencePiece tokenizer on the full CodeSearchNet-Java corpus, then runs T5 span-corruption pretraining for 10 epochs, checkpointing at epochs 1/3/10 (`configs/pretrain_10ep.yaml`). See `docs/gpu-reproduction.md` for the full launch order and what to bring back. Pre-launch check: `pop smoke` should already be passing on your local machine before you spend GPU time here.

### GPU check

Make sure Colab gave you a GPU runtime (Runtime > Change runtime type > GPU) before continuing.

In [ ]:
!nvidia-smi

### Install

This clones the repo at branch `main` and installs `pop` from it. **Pin the exact commit
you actually want to run before launching** -- edit the `git clone` cell below to
`git clone --depth 1 <repo> repo && cd repo && git checkout <commit-sha>`, or swap the branch
name for a tag/commit once the code is on `main`. Running against a moving branch tip means
your results may not match what you reviewed in the PR.

In [ ]:
!git clone --depth 1 -b main https://github.com/yib7/Strats-for-Bug-Fixing.git repo
%cd repo
%pip install -q -e .

### Weights & Biases login

Run the cell below and paste your own W&B API key when prompted (interactive login -- your key
is never stored in this notebook or read by anyone else). If you skip this cell, training still
runs; `pop` auto-disables W&B reporting when `WANDB_API_KEY` isn't set (see
`pop.train.pretrain`/`pop.train.finetune`).

In [ ]:
import wandb

wandb.login()

### Train tokenizer + run pretraining

Trains the tokenizer to `outputs/tokenizer/tokenizer.model` (referenced by `configs/pretrain_10ep.yaml`'s `tokenizer_path`), then runs `pop pretrain`. This is a single long-running cell; expect it to take a while on a T4 (see `docs/gpu-reproduction.md` for the runtime estimate).

In [ ]:
from pop.tokenizer.train import train_tokenizer
from pop.data.corpus import load_pretraining_corpus

corpus = load_pretraining_corpus(50000, seed=42)
train_tokenizer(corpus, "outputs/tokenizer/tokenizer.model", vocab_size=16384)

!pop pretrain --config configs/pretrain_10ep.yaml

### Download results

Bring the results JSON (and, for pretrain/finetune, the checkpoint directories under
`outputs/`) back to your local clone -- see `docs/gpu-reproduction.md` for exactly where each phase's
artifacts belong.

This phase writes checkpoints, not a `results/*.json` -- zip and download `outputs/pretrain/` (or at least `outputs/pretrain/final/` and any epoch-1/3 checkpoints you want) and `outputs/tokenizer/`; you'll need both for `colab_finetune.ipynb`.

In [ ]:
from google.colab import files

files.download("outputs/pretrain/final")  # noqa: adjust/comment out paths you don't need